# Pipeline de Modelado Base — Dataset Aumentado BioCatch (~1M sesiones)

**Iniciativa Anti-Vishing:** Evaluacion de modelos base sobre el dataset aumentado sinteticamente.

## Objetivo

Entrenar y evaluar cuatro algoritmos de clasificacion (Regresion Logistica, Random Forest, XGBoost y Redes Neuronales) sobre el dataset aumentado `biocatch_augmented_1M.parquet`.
A diferencia del pipeline anterior (`3_Modeling_Vishing_Pipeline.ipynb`), aqui no se iteran variantes de balanceo ya que el dataset es unico.

**Diferencias clave vs pipeline original:**
- Dataset unico (sin iteracion sobre tecnicas de balanceo)
- Imbalance severo: 1:94 (vs 1:19 en el dataset original)
- Nuevas features de sensores y comportamiento: `errors_per_minute`, `interactions_per_s`, `hesitation_composite`, `avg_touch_size_px`, `device_tilt_variability`, `gyro_rotation_rate_mean`, `accelerometer_jerk_mean`, `unique_screens_visited`
- Sin scores BioCatch (no disponibles en el dataset aumentado)

**Manejo del desbalance:**
- Submuestreo estratificado del training set (140K legitimas + todos los vishing)
- `class_weight="balanced"` para LR y RF
- `scale_pos_weight` para XGBoost
- `sample_weight` para MLP

**Metrica prioritaria:** PR-AUC y Recall (contexto de deteccion de fraude con imbalance 1:94)

## 1. Setup e Importaciones

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_curve
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Librerias cargadas correctamente.')


Librerias cargadas correctamente.


## 2. Carga del Dataset Aumentado

El dataset `biocatch_augmented_1M.parquet` contiene ~1M sesiones sinteticas generadas a partir del dataset original de 50K sesiones.

| Aspecto | Original (50K) | Aumentado (~1M) |
|---------|----------------|------------------|
| Filas | 50,000 | 1,001,940 |
| Features usables | 47 | 47 |
| Vishing (%) | 5.0% | ~1.05% |
| Imbalance ratio | 1:19 | 1:94 |
| Scores BioCatch | Si | No |
| Features nuevas | — | 8 (sensores + engineered) |

**Features nuevas en el dataset aumentado:** `errors_per_minute`, `interactions_per_s`, `hesitation_composite`, `avg_touch_size_px`, `device_tilt_variability`, `gyro_rotation_rate_mean`, `accelerometer_jerk_mean`, `unique_screens_visited`

In [2]:
PARQUET_PATH = 'data_augmentation/biocatch_augmented_1M.parquet'

df = pd.read_parquet(PARQUET_PATH)
df['session_timestamp'] = pd.to_datetime(df['session_timestamp'])

n_total   = len(df)
n_vishing = int((df['is_vishing'] == 1).sum())
n_legit   = int((df['is_vishing'] == 0).sum())
imbalance_ratio = n_legit // n_vishing

print(f'Dataset aumentado : {n_total:,} filas x {df.shape[1]} columnas')
print(f'  Legitimas       : {n_legit:,}  ({n_legit/n_total*100:.2f}%)')
print(f'  Vishing         : {n_vishing:,}   ({n_vishing/n_total*100:.2f}%)')
print(f'  Imbalance ratio : 1:{imbalance_ratio}')
print()
print(f'Rango temporal : {df["session_timestamp"].min().date()} -> {df["session_timestamp"].max().date()}')
print()
print(f'Columnas ({df.shape[1]}):')
for col in sorted(df.columns):
    print(f'  {col}: {df[col].dtype}')


Dataset aumentado : 1,001,940 filas x 54 columnas
  Legitimas       : 991,440  (98.95%)
  Vishing         : 10,500   (1.05%)
  Imbalance ratio : 1:94

Rango temporal : 2024-06-01 -> 2025-05-31

Columnas (54):
  accelerometer_jerk_mean: float32
  amount_field_corrections: int32
  app_version: category
  avg_hesitation_duration_s: float32
  avg_interkey_latency_ms: float32
  avg_keyhold_ms: float32
  avg_touch_pressure: float32
  avg_touch_size_px: float32
  beneficiary_field_corrections: int32
  call_overlap_duration_s: float32
  copy_paste_events: int32
  customer_id: category
  data_familiarity_score: float32
  dead_time_periods: int32
  dead_time_ratio: float32
  device_tilt_angle_mean: float32
  device_tilt_variability: float32
  device_type: category
  doodling_events: int32
  errors_per_minute: float32
  gyro_rotation_rate_mean: float32
  hesitation_composite: float32
  hesitation_count: float32
  hour_of_day: int32
  input_correction_count: int32
  input_error_count: int32
  inte

## 3. Preparacion del Dataset y Extraccion del Hold-out

### Columnas eliminadas
- **Identificadores** (`session_id`, `customer_id`, `session_timestamp`): sin valor predictivo
- **Variables categoricas de dispositivo** (`device_type`, `os_type`, `app_version`): mismo criterio que el pipeline original

### Estrategia de entrenamiento para 1M filas

**Hold-out (20%):** ~200K filas estratificadas, virgenes para evaluacion final.

**Training set (subsample del 80%):** Para hacer el entrenamiento computacionalmente factible:
- Todos los vishing del 80% de train (~8,400 filas)
- 140,000 legitimas del 80% de train
- Total: ~148,400 filas con ratio ~1:16

> El desbalance residual 1:16 en entrenamiento se gestiona via `class_weight`/`scale_pos_weight`.
> Este enfoque es coherente con el analisis EDA (notebook 5) que uso la misma proporcion para Random Forest.


In [4]:
COLS_ID = ['session_id', 'customer_id', 'session_timestamp',
           'device_type', 'os_type', 'app_version']

df_model = df.drop(columns=[c for c in COLS_ID if c in df.columns])

X = df_model.drop(columns=['is_vishing'])
y = df_model['is_vishing']

# ── Hold-out: 20% estratificado (virgen) ──
X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('HOLD-OUT SET (20% del total — virgen)')
print(f'  Total     : {len(X_test):,}')
print(f'  Legitimas : {(y_test==0).sum():,} ({(y_test==0).mean()*100:.2f}%)')
print(f'  Vishing   : {y_test.sum():,} ({y_test.mean()*100:.2f}%)')

# ── Training set: subsample del 80% ──
df_train_pool = X_train_pool.copy()
df_train_pool['is_vishing'] = y_train_pool.values

df_vishing_tr = df_train_pool[df_train_pool['is_vishing'] == 1]
df_legit_tr   = df_train_pool[df_train_pool['is_vishing'] == 0]

N_LEGIT_TRAIN = 140_000
df_train = pd.concat([
    df_legit_tr.sample(N_LEGIT_TRAIN, random_state=42),
    df_vishing_tr
]).sample(frac=1, random_state=42).reset_index(drop=True)

X_train = df_train.drop(columns=['is_vishing'])
y_train = df_train['is_vishing']

ratio_train = (y_train == 0).sum() / y_train.sum()

print()
print('SET DE ENTRENAMIENTO (subsample del 80%)')
print(f'  Total     : {len(X_train):,}')
print(f'  Legitimas : {(y_train==0).sum():,} ({(y_train==0).mean()*100:.2f}%)')
print(f'  Vishing   : {y_train.sum():,} ({y_train.mean()*100:.2f}%)')
print(f'  Ratio     : 1:{ratio_train:.1f}')

# Escalado para LR y MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

FEATURE_NAMES = X_train.columns.tolist()
print(f'\nFeatures de entrada ({len(FEATURE_NAMES)}):')
print(FEATURE_NAMES)


HOLD-OUT SET (20% del total — virgen)
  Total     : 200,388
  Legitimas : 198,288 (98.95%)
  Vishing   : 2,100 (1.05%)

SET DE ENTRENAMIENTO (subsample del 80%)
  Total     : 148,400
  Legitimas : 140,000 (94.34%)
  Vishing   : 8,400 (5.66%)
  Ratio     : 1:16.7

Features de entrada (47):
['avg_keyhold_ms', 'avg_interkey_latency_ms', 'typing_speed_cps', 'keystroke_variability', 'segmented_typing_ratio', 'avg_touch_pressure', 'avg_touch_size_px', 'swipe_speed_px_s', 'swipe_directional_variance', 'scroll_speed_avg', 'device_tilt_angle_mean', 'device_tilt_variability', 'gyro_rotation_rate_mean', 'accelerometer_jerk_mean', 'phone_motion_events', 'hesitation_count', 'avg_hesitation_duration_s', 'max_hesitation_duration_s', 'dead_time_periods', 'total_dead_time_s', 'dead_time_ratio', 'screens_visited', 'unique_screens_visited', 'unusual_screen_visits', 'navigation_back_count', 'screen_transition_time_avg_s', 'input_error_count', 'input_correction_count', 'amount_field_corrections', 'benefici

## 4. Definicion de Modelos Base

Los mismos 4 algoritmos del pipeline original, adaptados para el desbalance severo del dataset aumentado.

| Modelo | Manejo del Desbalance | Requiere Escalado |
|--------|-----------------------|-------------------|
| Logistic Regression | `class_weight="balanced"` | Si |
| Random Forest | `class_weight="balanced"` | No |
| XGBoost | `scale_pos_weight=ratio_train` (~16x) | No |
| Deep Learning (MLP) | `sample_weight` (inverso de frecuencia de clase) | Si |

In [5]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150, max_depth=10, random_state=42,
        class_weight='balanced', n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        use_label_encoder=False, eval_metric='logloss',
        max_depth=6, learning_rate=0.1, random_state=42,
        scale_pos_weight=ratio_train, n_jobs=-1
    ),
    'Deep Learning (MLP)': MLPClassifier(
        hidden_layer_sizes=(64, 32), activation='relu',
        solver='adam', max_iter=300, random_state=42
    )
}

sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

print('Modelos definidos:')
for name in models:
    print(f'  - {name}')
print(f'\nscale_pos_weight (XGBoost) : {ratio_train:.1f}')
print(f'Features de entrada        : {len(FEATURE_NAMES)}')
print(f'Filas de entrenamiento     : {len(X_train):,}')
print(f'Filas hold-out             : {len(X_test):,}')


Modelos definidos:
  - Logistic Regression
  - Random Forest
  - XGBoost
  - Deep Learning (MLP)

scale_pos_weight (XGBoost) : 16.7
Features de entrada        : 47
Filas de entrenamiento     : 148,400
Filas hold-out             : 200,388


## 5. Entrenamiento y Evaluacion

Cada modelo se entrena sobre el training set (~148K filas) y se evalua sobre el hold-out virgen (~200K filas).

**Metricas reportadas:**
- **Recall**: fraccion de casos vishing detectados — critica en deteccion de fraude
- **Precision**: de los alertados, cuantos son realmente vishing
- **F1**: balance armonico entre recall y precision
- **ROC-AUC**: poder discriminativo global
- **PR-AUC**: metrica principal para datasets desbalanceados (area bajo curva Precision-Recall)


In [ ]:
results = []

for model_name, model in models.items():
    print(f'Entrenando: {model_name}...', end=' ', flush=True)

    use_scaled = model_name in ['Logistic Regression', 'Deep Learning (MLP)']
    X_tr = X_train_scaled if use_scaled else X_train.values
    X_te = X_test_scaled  if use_scaled else X_test.values

    if model_name == 'Deep Learning (MLP)':
        model.fit(X_tr, y_train, sample_weight=sample_weights_train)
    else:
        model.fit(X_tr, y_train)

    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    recall    = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)
    roc_auc   = roc_auc_score(y_test, y_prob)
    pr_auc    = average_precision_score(y_test, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    results.append({
        'Modelo'    : model_name,
        'Recall'    : recall,
        'Precision' : precision,
        'F1'        : f1,
        'ROC_AUC'   : roc_auc,
        'PR_AUC'    : pr_auc,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn
    })

    print(f'Recall={recall:.4f} | Precision={precision:.4f} | PR_AUC={pr_auc:.4f}')

print('\nEntrenamiento completado.')


Entrenando: Logistic Regression... Recall=0.7567 | Precision=0.0475 | PR_AUC=0.3729
Entrenando: Random Forest... Recall=0.7495 | Precision=0.1428 | PR_AUC=0.6141
Entrenando: XGBoost... Recall=0.8424 | Precision=0.0614 | PR_AUC=0.6414
Entrenando: Deep Learning (MLP)... 

## 6. Tabla Comparativa de Resultados

Ordenado por PR-AUC — metrica mas informativa para datasets con imbalance severo.


In [ ]:
df_results = pd.DataFrame(results)
df_results_sorted = df_results.sort_values('PR_AUC', ascending=False).reset_index(drop=True)

display(
    df_results_sorted[['Modelo', 'Recall', 'Precision', 'F1', 'ROC_AUC', 'PR_AUC']]
    .style
    .format({'Recall': '{:.4f}', 'Precision': '{:.4f}', 'F1': '{:.4f}',
             'ROC_AUC': '{:.4f}', 'PR_AUC': '{:.4f}'})
    .background_gradient(cmap='viridis', subset=['Recall', 'PR_AUC', 'ROC_AUC'])
    .set_caption('Resultados en Hold-out Virgen (20%, ~200K filas) — Ordenado por PR-AUC')
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_cfg = [('Recall', '#e74c3c', 'Recall (Sensibilidad)'),
               ('PR_AUC', '#3498db', 'PR-AUC (Metrica Prioritaria)'),
               ('ROC_AUC', '#2ecc71', 'ROC-AUC')]

for ax, (metric, color, title) in zip(axes, metrics_cfg):
    df_plot = df_results.sort_values(metric, ascending=True)
    bars = ax.barh(df_plot['Modelo'], df_plot[metric], color=color, alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel(metric)
    ax.set_xlim(0, 1.12)
    for bar, val in zip(bars, df_plot[metric]):
        ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Comparacion de Modelos Base — Dataset Aumentado BioCatch',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 7. Analisis del Mejor Modelo (Matriz de Confusion)

Se identifica el mejor modelo por PR-AUC y se analiza la matriz de confusion sobre el hold-out.

- **Falsos Negativos (FN):** casos vishing no detectados — **riesgo critico en fraude financiero**
- **Falsos Positivos (FP):** sesiones legitimas marcadas como vishing — costo operacional (revision manual)


In [ ]:
best_name = df_results_sorted.iloc[0]['Modelo']
print(f'Mejor modelo (PR-AUC): {best_name}')
print(f"  PR-AUC   : {df_results_sorted.iloc[0]['PR_AUC']:.4f}")
print(f"  ROC-AUC  : {df_results_sorted.iloc[0]['ROC_AUC']:.4f}")
print(f"  Recall   : {df_results_sorted.iloc[0]['Recall']:.4f}")
print(f"  Precision: {df_results_sorted.iloc[0]['Precision']:.4f}")

best_model  = models[best_name]
use_scaled  = best_name in ['Logistic Regression', 'Deep Learning (MLP)']
X_te_best   = X_test_scaled if use_scaled else X_test.values

y_pred_best = best_model.predict(X_te_best)
y_prob_best = best_model.predict_proba(X_te_best)[:, 1]

cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legitimo (pred)', 'Vishing (pred)'],
            yticklabels=['Legitimo (real)', 'Vishing (real)'])
axes[0].set_title('Matriz de Confusion — Conteos', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Clase Real')
axes[0].set_xlabel('Clase Predicha')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Legitimo (pred)', 'Vishing (pred)'],
            yticklabels=['Legitimo (real)', 'Vishing (real)'])
axes[1].set_title('Matriz de Confusion — Normalizada (Recall por clase)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Clase Real')
axes[1].set_xlabel('Clase Predicha')

plt.suptitle(f'{best_name}  |  Dataset Aumentado BioCatch  |  Hold-out set', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n{'='*57}")
print(f"  Verdaderos Negativos (TN) : {tn:>6,}  — Legitimos identificados correctamente")
print(f"  Falsos Positivos     (FP) : {fp:>6,}  — Legitimos marcados como Vishing")
print(f"  Falsos Negativos     (FN) : {fn:>6,}  — Vishing no detectados  <-- riesgo critico")
print(f"  Verdaderos Positivos (TP) : {tp:>6,}  — Vishing detectados correctamente")
print(f"{'='*57}")
print(f"  Recall    (Sensibilidad) : {tp/(tp+fn):.4f}")
print(f"  Precision                : {tp/(tp+fp):.4f}")
print(f"  F1-Score                 : {2*tp/(2*tp+fp+fn):.4f}")
print(f"  ROC-AUC                  : {roc_auc_score(y_test, y_prob_best):.4f}")
print(f"  PR-AUC                   : {average_precision_score(y_test, y_prob_best):.4f}")
print(f"{'='*57}")


## 8. Feature Importance — XGBoost

XGBoost ofrece interpretabilidad nativa via importancia de variables por Gain acumulado.
Este analisis es independiente del modelo ganador y valida los hallazgos del EDA.


In [ ]:
xgb_model = models['XGBoost']
fi = pd.DataFrame({
    'feature'   : FEATURE_NAMES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 10))
top_fi = fi.head(25)

q80 = top_fi['importance'].quantile(0.80)
q60 = top_fi['importance'].quantile(0.60)
colors_fi = ['#e74c3c' if v >= q80 else '#f39c12' if v >= q60 else '#3498db'
             for v in top_fi['importance']]

bars = ax.barh(range(len(top_fi)), top_fi['importance'].values,
               color=colors_fi, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(top_fi)))
ax.set_yticklabels(top_fi['feature'].values)
ax.set_xlabel('Importancia (Gain acumulado)')
ax.set_title('Feature Importance — XGBoost  |  Top 25 Features', fontweight='bold', fontsize=14)
ax.invert_yaxis()
for i, v in enumerate(top_fi['importance'].values):
    ax.text(v + top_fi['importance'].max()*0.005, i, f'{v:.4f}', va='center', fontsize=8)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#e74c3c', label='Top 20% importancia'),
    Patch(facecolor='#f39c12', label='Top 20-40%'),
    Patch(facecolor='#3498db', label='Resto'),
], loc='lower right')

plt.tight_layout()
plt.show()

print('Top 15 features (XGBoost — Gain):')
print(f"{'Feature':40s} {'Importancia':>12s}")
print('-' * 55)
for _, row in fi.head(15).iterrows():
    print(f"{row['feature']:40s} {row['importance']:12.4f}")


## 9. Curva Precision-Recall y Analisis de Umbral de Decision

Con un imbalance de 1:94, el umbral por defecto (0.5) suele no ser optimo.
Analizamos el tradeoff Precision-Recall del mejor modelo en funcion del umbral.

**Guia practica:**
- Umbral **bajo** → alto Recall, menor Precision (mas alertas, mas falsos positivos — mayor cobertura)
- Umbral **alto** → alta Precision, menor Recall (menos alertas, mas vishing perdidos)
- El umbral optimo por F1 es un punto de equilibrio; en produccion puede ajustarse segun tolerancia operacional


In [ ]:
precisions_c, recalls_c, thresholds_c = precision_recall_curve(y_test, y_prob_best)
pr_auc_val = average_precision_score(y_test, y_prob_best)

f1_scores_c = (2 * precisions_c[:-1] * recalls_c[:-1]
               / (precisions_c[:-1] + recalls_c[:-1] + 1e-9))
best_idx  = np.argmax(f1_scores_c)
best_thr  = thresholds_c[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Curva PR
axes[0].plot(recalls_c, precisions_c, color='#3498db', linewidth=2,
             label=f'PR-AUC = {pr_auc_val:.4f}')
axes[0].axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.7,
                label=f'Baseline sin modelo ({y_test.mean():.4f})')
axes[0].fill_between(recalls_c, precisions_c, alpha=0.08, color='#3498db')
axes[0].scatter([recalls_c[best_idx]], [precisions_c[best_idx]], s=100,
                color='red', zorder=5, label=f'Optimo F1 (thr={best_thr:.3f})')
axes[0].set_xlabel('Recall (Sensibilidad)', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title(f'Curva Precision-Recall — {best_name}', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

# Metricas vs umbral
axes[1].plot(thresholds_c, precisions_c[:-1], label='Precision', color='#2ecc71', linewidth=2)
axes[1].plot(thresholds_c, recalls_c[:-1],    label='Recall',    color='#e74c3c', linewidth=2)
axes[1].plot(thresholds_c, f1_scores_c,       label='F1',        color='#f39c12', linewidth=2, linestyle='--')
axes[1].axvline(x=best_thr, color='black', linestyle=':', alpha=0.8,
                label=f'Optimo F1 = {best_thr:.3f}')
axes[1].axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Default (0.5)')
axes[1].set_xlabel('Umbral de Decision', fontsize=12)
axes[1].set_ylabel('Metrica', fontsize=12)
axes[1].set_title('Precision, Recall y F1 vs Umbral', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)

plt.suptitle(f'Analisis del Umbral de Decision — {best_name}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

y_pred_opt = (y_prob_best >= best_thr).astype(int)

print(f"{'='*62}")
print(f"  Con umbral optimo (max F1) = {best_thr:.4f}")
print(f"    Recall    : {recall_score(y_test, y_pred_opt):.4f}")
print(f"    Precision : {precision_score(y_test, y_pred_opt):.4f}")
print(f"    F1        : {f1_score(y_test, y_pred_opt):.4f}")
print(f"{'='*62}")
print(f"  Con umbral por defecto = 0.5")
print(f"    Recall    : {recall_score(y_test, y_pred_best):.4f}")
print(f"    Precision : {precision_score(y_test, y_pred_best):.4f}")
print(f"    F1        : {f1_score(y_test, y_pred_best):.4f}")
print(f"{'='*62}")
